In [4]:
import os, sys, re, gzip, io, json
from datetime import datetime
import pandas as pd

In [5]:
# ---------- config from environment (00_config.sh sets RUN/REPO_ROOT/OUT_DIR/META_DIR) ----------
REPO_ROOT = os.environ.get("REPO_ROOT", os.getcwd())
RUN       = os.environ.get("RUN", "genotype_run1")
OUT_DIR   = os.environ.get("OUT_DIR", os.path.join(REPO_ROOT, "output", RUN))
META_DIR  = os.environ.get("META_DIR", os.path.join(REPO_ROOT, "metadata"))

In [66]:
SAMPLE_SHEET = os.path.join(REPO_ROOT, "input_data", "sample_sheet")
EXP_INDIR = os.path.join(REPO_ROOT, "input_data", "expression_microarray")
EXP_OUT   = os.path.join(OUT_DIR, "expr", "explore")
os.makedirs(EXP_OUT, exist_ok=True)

In [69]:
# creates a dictionary to keep the required files
FILES = {
    "tga_txt":       os.path.join(EXP_INDIR, "TGA-cohort.txt"),
    "tga_csv":       os.path.join(EXP_INDIR, "TGA-cohort-BackUp.csv"),
    "transpose_csv": os.path.join(EXP_INDIR, "transpose_numbers.csv"),
    "snp_csv": os.path.join(SAMPLE_SHEET, "All_samples_Examine_SNPs_GWAS studies_GJ-P01_infiniumSampleSheet.csv")
}

In [8]:
import os

tga_txt = FILES["tga_txt"]
assert os.path.exists(tga_txt), f"Missing file: {tga_txt}"

print("Path:", tga_txt)
print("Size (MB):", round(os.path.getsize(tga_txt)/1e6, 3))

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/TGA-cohort.txt
Size (MB): 39.302


In [ ]:
# Peek first ~10 non-empty lines raw (helps see delimiter/headers)
with open(tga_txt, "r", encoding="utf-8", errors="replace") as f:
    for i in range(10):
        line = f.readline()
        if not line: break
        print(repr(line.rstrip("\n")))

In [9]:
import subprocess
subprocess.run(["code", "--reuse-window", tga_txt])

CompletedProcess(args=['code', '--reuse-window', '/home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/TGA-cohort.txt'], returncode=0)

In [11]:
# Load with pandas
# Try auto-delimiter first; if it misguesses, we’ll force sep="\t" next cell
df_txt = pd.read_csv(tga_txt, sep="\t", engine="c", low_memory=False)
display(df_txt.head(5))
print("shape:", df_txt.shape)

,Original,Summarized,Difference,Filename,Chip Type,Scan Date,UASG,Final Diagnosis,Diagnosis Subtype,Sex,...,TSUnmapped00000810.hg.1,TSUnmapped00000812.hg.1,TSUnmapped00000815.hg.1,TSUnmapped00000817.hg.1,TSUnmapped00000818.hg.1,TSUnmapped00000819.hg.1,TSUnmapped00000820.hg.1,TSUnmapped00000821.hg.1,TSUnmapped00000822.hg.1,TSUnmapped00000823.hg.1
0,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL,Clariom_S_Human_HT,07/03/2020,UASG-1467,Ischemic Stroke,Large vessel disease,Male,...,2.49253,2.41132,3.33450,2.13259,2.57130,8.51848,2.02642,2.83986,2.45988,4.66323
1,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0162,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,2.52214,2.58208,2.90031,2.25478,2.70673,8.49474,1.66142,2.42892,2.67258,4.67467
2,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0181,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,2.53402,2.37989,3.35462,1.67938,2.64475,8.84778,1.78275,3.62172,3.27673,4.68155
3,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-1127,Ischemic Stroke,Other,Female,...,2.26048,2.47972,3.27244,2.05983,2.54761,8.18061,1.73277,3.27272,2.62022,4.24615
4,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0007,Ischemic Stroke,Cardioembolic,Male,...,2.91858,2.34173,3.55027,2.35425,2.69594,8.58109,1.96196,3.54381,3.14377,3.66505


shape: (229, 21621)


In [12]:
print("Shape:", df_txt.shape)

Shape: (229, 21621)


In [18]:
print("First 12 column names:", list(df_txt.columns[:12]))

First 12 column names: ['Original', 'Summarized', 'Difference', 'Filename', 'Chip Type', 'Scan Date', 'UASG', 'Final Diagnosis', 'Diagnosis Subtype', 'Sex', 'Age At Onset', 'Hypertension']


In [28]:
print("First 175 column names:", list(df_txt.columns[:175]))

First 175 column names: ['Original', 'Summarized', 'Difference', 'Filename', 'Chip Type', 'Scan Date', 'UASG', 'Final Diagnosis', 'Diagnosis Subtype', 'Sex', 'Age At Onset', 'Hypertension', 'Diabetes', 'Hypercholesterolemia', 'Insulin', 'Oral diabetes meds', 'Insulin.1', 'GLUCOSE', 'GLUCOSE2', 'GLUCOSE FASTING', 'HEMOGLOBIN A1C', 'WHITE BLOOD CELL COUNT', 'HEMOGLOBIN', 'HEMATOCRIT', 'PLATELET COUNT', 'BLOOD UREA', 'CREATININE BLOOD', 'ESTIMATED GFR', 'GLUCOSE.1', 'INR', 'aPTT', 'NEUTROPHIL PERCENT', 'LYMPHOCYTE PERCENT', 'MONOCYTE PERCENT', 'WHITE BLOOD CELL COUNT2', 'HEMOGLOBIN2', 'HEMATOCRIT2', 'PLATELET COUNT2', 'BLOOD UREA2', 'CREATININE BLOOD2', 'ESTIMATED GFR2', 'GLUCOSE2.1', 'INR2', 'aPTT2', 'NEUTROPHIL PERCENT2', 'LYMPHOCYTE PERCENT2', 'MONOCYTE PERCENT2', 'CHOLESTEROL', 'HDL', 'LDL', 'TRIGLYCERIDE', 'Anticoagulation', 'Final Diagnosis.1', 'Ischemic stroke/ TIA cause', 'PmHx-Stroke', 'Congestive heart failure', 'Chronic renal insufficiency', 'Cancer', 'Unusual Disese', 'tPA', '

In [17]:
print("Last 5 column names:", list(df_txt.columns[-5:]))

Last 5 column names: ['TSUnmapped00000819.hg.1', 'TSUnmapped00000820.hg.1', 'TSUnmapped00000821.hg.1', 'TSUnmapped00000822.hg.1', 'TSUnmapped00000823.hg.1']


In [23]:
display(df_txt.iloc[:, :12].head(3))

,Original,Summarized,Difference,Filename,Chip Type,Scan Date,UASG,Final Diagnosis,Diagnosis Subtype,Sex,Age At Onset,Hypertension
0,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL,Clariom_S_Human_HT,07/03/2020,UASG-1467,Ischemic Stroke,Large vessel disease,Male,71,Yes
1,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0162,Ischemic Stroke,Small vessel disease / Lacunar,Male,35,No
2,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0181,Ischemic Stroke,Small vessel disease / Lacunar,Male,55,Yes


------------------------

In [ ]:
# Peek at raw text to guess the delimiter + header row
with open(tga_csv, "rb") as f:         # open the file whose path is in tga_csv, read as bytes (not regular text)
    for i in range(5):                 # look at first 5 lines
        print(f.readline().decode("utf-8", "replace").rstrip("\n")) # turn those bytes into text using UTF-8

        # If the file has any weird characters that don’t fit UTF-8, "replace" puts a special � symbol instead of crashing.
        # .rstrip("\n") --“Remove the newline character at the end,” so the printed lines don’t have an extra blank line between them.

In [33]:
import os

tga_csv = FILES["tga_csv"] # Pulls a path string from a dict called FILES under the key "tga_csv"
assert os.path.exists(tga_csv), f"Missing file: {tga_csv}" # Stops the program if the file doesn’t exist.
print("Path:", tga_csv)  #Prints the file path
print("Size (MB):", round(os.path.getsize(tga_csv)/1e6, 3)) #Prints the file size

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/TGA-cohort-BackUp.csv
Size (MB): 39.303


In [43]:
import pandas as pd

df_csv = pd.read_csv(
    tga_csv,
    sep=None,               # auto-detect delimiter
    engine="python",        # needed for sep=None or regex seps
    encoding="utf-8",       # try utf-8 first; fallback to latin-1 if errors
    on_bad_lines="warn"     # show malformed lines instead of crashing
)
print(df_csv.shape)
df_csv.head(5)


(229, 21628)


,Original,Summarized,Difference,Filename,Chip Type,Scan Date,UASG,Final Diagnosis,Diagnosis Subtype,Sex,...,TSUnmapped00000810.hg.1,TSUnmapped00000812.hg.1,TSUnmapped00000815.hg.1,TSUnmapped00000817.hg.1,TSUnmapped00000818.hg.1,TSUnmapped00000819.hg.1,TSUnmapped00000820.hg.1,TSUnmapped00000821.hg.1,TSUnmapped00000822.hg.1,TSUnmapped00000823.hg.1
0,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL.pimg,9SR35159A_G04.CEL,Clariom_S_Human_HT,07/03/2020,UASG-1467,Ischemic Stroke,Large vessel disease,Male,...,2.83986,2.45988,4.66323,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL.pimg,9SR35157A_A09.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0162,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,2.42892,2.67258,4.67467,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL.pimg,9SR35157A_G11.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0181,Ischemic Stroke,Small vessel disease / Lacunar,Male,...,3.62172,3.27673,4.68155,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL.pimg,9SR35157A_D05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-1127,Ischemic Stroke,Other,Female,...,3.27272,2.62022,4.24615,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL.pimg,9SR35157A_F05.CEL,Clariom_S_Human_HT,06/30/2020,UASG-0007,Ischemic Stroke,Cardioembolic,Male,...,3.54381,3.14377,3.66505,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
# how many unique IDs ins UASG column
print("UASG" in df_csv.columns)  
print(df_csv["UASG"].head())

# Count total vs unique
total_ids = df_csv["UASG"].shape[0]
unique_ids = df_csv["UASG"].nunique()
print(f"total_ids: {total_ids}")
print(f"unique_ids: {unique_ids}")

True
0    UASG-1467
1    UASG-0162
2    UASG-0181
3    UASG-1127
4    UASG-0007
Name: UASG, dtype: object
total_ids: 229
unique_ids: 229


In [59]:
print(df_csv["Final Diagnosis"].head())

0    Ischemic Stroke
1    Ischemic Stroke
2    Ischemic Stroke
3    Ischemic Stroke
4    Ischemic Stroke
Name: Final Diagnosis, dtype: object


In [60]:
# Count unique categories
n_categories = df_csv["Final Diagnosis"].nunique()
print("Number of unique Final Diagnosis categories:", n_categories)

Number of unique Final Diagnosis categories: 5


In [61]:
# List them all with counts
print(df_csv["Final Diagnosis"].value_counts())

Final Diagnosis
Ischemic Stroke       170
Control                54
TIA                     3
Hemorrhagic Stroke      1
TIA vs Control          1
Name: count, dtype: int64


In [62]:
# If you only want the unique category names:
print(df_csv["Final Diagnosis"].unique())

['Ischemic Stroke' 'Control' 'Hemorrhagic Stroke' 'TIA' 'TIA vs Control']


-----------

In [44]:
import os

transpose_csv = FILES["transpose_csv"] # Pulls a path string from a dict called FILES under the key "tga_csv"
assert os.path.exists(transpose_csv), f"Missing file: {transpose_csv}" # Stops the program if the file doesn’t exist.
print("Path:", transpose_csv)  #Prints the file path
print("Size (MB):", round(os.path.getsize(transpose_csv)/1e6, 3)) #Prints the file size

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/expression_microarray/transpose_numbers.csv
Size (MB): 46.415


In [45]:
import pandas as pd

df_transpose = pd.read_csv(
    transpose_csv,
    sep=None,               # auto-detect delimiter
    engine="python",        # needed for sep=None or regex seps
    encoding="utf-8",       # try utf-8 first; fallback to latin-1 if errors
    on_bad_lines="warn"     # show malformed lines instead of crashing
)
print(df_transpose.shape)
df_transpose.head(5)


(21449, 234)


,ID,Gene Symbol,unigene,gene_assignment,RefSeq,swissprot,unigene.1,UASG-0432,UASG-1331,UASG-1226,...,UASG-0202,UASG-0221,UASG-0304,UASG-0342,UASG-0361,UASG-1027,UASG-1155,UASG-1325,UASG-1420,UASG-0220
0,mRS good vs bad,NaN,NaN,NaN,NaN,NaN,NaN,1.00000,1.00000,1.00000,...,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000
1,TC1000010642.hg.1,A1CF,NM_001198818 // Hs.282795 // connective tissue...,NM_001198818 // A1CF // APOBEC1 complementatio...,NM_001198818,NM_001198818 // Q9NQ94 /// NM_001198818 // Q7Z...,NM_001198818 // Hs.282795 // connective tissue...,1.62489,1.94362,1.84973,...,1.46509,1.55474,1.51391,1.53876,1.72148,1.61030,1.88767,1.84530,1.61760,1.64039
2,TC1600006789.hg.1,A2BP1,---,A2BP1.qAug10-unspliced // A2BP1 // Transcript ...,A2BP1.qAug10-unspliced,---,---,1.73784,1.31067,1.53732,...,1.12328,1.43262,1.23415,1.11851,1.23606,1.63694,1.27302,1.51636,1.27112,1.33102
3,TC1600006790.hg.1,A2BP1,---,A2BP1.tAug10-unspliced // A2BP1 // Transcript ...,A2BP1.tAug10-unspliced,---,---,1.93455,1.80662,1.91982,...,1.83373,1.67877,2.28307,1.77295,1.68748,1.77301,1.83773,1.87122,2.70517,1.57111
4,TC1600006812.hg.1,A2BP1,---,A2BP1.sAug10-unspliced // A2BP1 // Transcript ...,A2BP1.sAug10-unspliced,---,---,2.78974,2.72053,3.19491,...,2.84769,3.14492,3.28217,2.97546,2.91276,3.02366,3.06257,2.92001,2.98420,2.49120


In [47]:
print("First 175 column names:", list(df_transpose.columns[:234]))

First 175 column names: ['ID', 'Gene Symbol', 'unigene', 'gene_assignment', 'RefSeq', 'swissprot', 'unigene.1', 'UASG-0432', 'UASG-1331', 'UASG-1226', 'UASG-1231', 'UASG-0119', 'UASG-0025', 'UASG-1185', 'UASG-1163', 'UASG-1230', 'UASG-1366', 'UASG-1563', 'UASG-1141', 'UASG-1126', 'UASG-1504', 'UASG-1318', 'UASG-1418', 'UASG-0115', 'UASG-1168', 'UASG-1110', 'UASG-1035', 'UASG-1381', 'UASG-1184', 'UASG-1374', 'UASG-1127', 'UASG-0038', 'UASG-0313', 'UASG-0154', 'UASG-0006', 'UASG-1109', 'UASG-1312', 'UASG-1412', 'UASG-1067', 'UASG-0049', 'UASG-1042', 'UASG-1291', 'UASG-0063', 'UASG-0271', 'UASG-0010', 'UASG-1061', 'UASG-0052', 'UASG-0092', 'UASG-0316', 'UASG-1187', 'UASG-0197', 'UASG-1450', 'UASG-1094', 'UASG-1056', 'UASG-0239', 'UASG-0153', 'UASG-1260', 'UASG-1259', 'UASG-1032', 'UASG-1030', 'UASG-1143', 'UASG-0134', 'UASG-0157', 'UASG-0234', 'UASG-1466', 'UASG-0144', 'UASG-1054', 'UASG-1069', 'UASG-1360', 'UASG-1404', 'UASG-0004', 'UASG-1052', 'UASG-1467', 'UASG-0162', 'UASG-0170', 'UAS

In [48]:
print("UASG columns", list(df_transpose.columns[7:234]))

First 175 column names: ['UASG-0432', 'UASG-1331', 'UASG-1226', 'UASG-1231', 'UASG-0119', 'UASG-0025', 'UASG-1185', 'UASG-1163', 'UASG-1230', 'UASG-1366', 'UASG-1563', 'UASG-1141', 'UASG-1126', 'UASG-1504', 'UASG-1318', 'UASG-1418', 'UASG-0115', 'UASG-1168', 'UASG-1110', 'UASG-1035', 'UASG-1381', 'UASG-1184', 'UASG-1374', 'UASG-1127', 'UASG-0038', 'UASG-0313', 'UASG-0154', 'UASG-0006', 'UASG-1109', 'UASG-1312', 'UASG-1412', 'UASG-1067', 'UASG-0049', 'UASG-1042', 'UASG-1291', 'UASG-0063', 'UASG-0271', 'UASG-0010', 'UASG-1061', 'UASG-0052', 'UASG-0092', 'UASG-0316', 'UASG-1187', 'UASG-0197', 'UASG-1450', 'UASG-1094', 'UASG-1056', 'UASG-0239', 'UASG-0153', 'UASG-1260', 'UASG-1259', 'UASG-1032', 'UASG-1030', 'UASG-1143', 'UASG-0134', 'UASG-0157', 'UASG-0234', 'UASG-1466', 'UASG-0144', 'UASG-1054', 'UASG-1069', 'UASG-1360', 'UASG-1404', 'UASG-0004', 'UASG-1052', 'UASG-1467', 'UASG-0162', 'UASG-0170', 'UASG-0262', 'UASG-0173', 'UASG-0189', 'UASG-1040', 'UASG-0257', 'UASG-0344', 'UASG-0005', 

---------------------------

In [71]:
import os

snp_csv = FILES["snp_csv"]                                 # Pulls a path string from a dict called FILES under the key "tga_csv"
assert os.path.exists(snp_csv), f"Missing file: {snp_csv}" # Stops the program if the file doesn’t exist.
print("Path:", snp_csv)  #Prints the file path
print("Size (MB):", round(os.path.getsize(snp_csv)/1e6, 3)) #Prints the file size

Path: /home/sima/git_projects/illumina-genotyping-pipeline/input_data/sample_sheet/All_samples_Examine_SNPs_GWAS studies_GJ-P01_infiniumSampleSheet.csv
Size (MB): 0.029


In [74]:
import pandas as pd

# --- read df_snp ---
df_snp = pd.read_csv(
    snp_csv,
    sep=None,             # auto-detect delimiter
    engine="python",      # required for sep=None
    encoding="utf-8",
    on_bad_lines="warn",
    skiprows=8            # make the 9th line the header row
)


In [75]:
# Normalize column names (trim stray spaces, etc.)
df_snp.columns = df_snp.columns.str.strip()

In [76]:
# Sanity check: make sure Sample_Name exists
if "Sample_Name" not in df_snp.columns:
    print("Columns seen:", list(df_snp.columns)[:20])
    raise KeyError("Expected 'Sample_Name' in header after skiprows=8")

print("df_snp shape:", df_snp.shape)
display(df_snp.head(5))

df_snp shape: (288, 11)


,Sample_Name,Sample_ID,Sample_Plate,Sample_Well,SentrixBarcode_A,SentrixPosition_A,Gender,Sample_Group,Replicate,Parent1,Parent2
0,NA10837,22684#NA10837#207363850074#R12C02,WG0203968-MSA3,H12,207363850074,R12C02,Male,50.00,NaN,NaN,NaN
1,UASG-0400,22684#UASG-0400#207339170112#R01C01,WG0203968-MSA3,E01,207339170112,R01C01,Female,76.61,NaN,NaN,NaN
2,UASG-0002,22684#UASG-0002#207363850013#R01C01,WG0203968-MSA3,A01,207363850013,R01C01,Female,68.18,NaN,NaN,NaN
3,UASG-0399,22684#UASG-0399#207363850013#R02C01,WG0203968-MSA3,A02,207363850013,R02C01,Male,61.75,NaN,NaN,NaN
4,UASG-0013,22684#UASG-0013#207363850013#R03C01,WG0203968-MSA3,A03,207363850013,R03C01,Female,70.06,NaN,NaN,NaN


In [77]:
snp_ids = (df_snp["Sample_Name"]
           .astype(str)                   # make every value a string (even NaN -> "nan")
           .str.strip()                   # trim leading/trailing spaces
           .replace({"": pd.NA,           # turn empty strings into NA
                     "nan": pd.NA,        # because astype(str) made NaN into "nan"
                     "None": pd.NA})      # common literal text for missing
           .dropna()                      # drop the missing values
           .unique())                     # keep one of each remaining ID


In [80]:
print("Unique UASG in df_snp:", len(snp_ids))

Unique UASG in df_snp: 288


In [86]:
# Count how many start with “UASG”
col = (df_snp["Sample_Name"]
       .astype(str)
       .str.strip())

mask_uasg = col.str.startswith("UASG", na=False)           # case-sensitive
# or case-insensitive:
# mask_uasg = col.str.match(r'(?i)^UASG', na=False)

n_total = col.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}).dropna().shape[0]
n_uasg  = mask_uasg.sum()
n_other = n_total - n_uasg

print("Total non-missing Sample_Name:", n_total)
print("Start with UASG:", n_uasg)
print("Other IDs:", n_other)


Total non-missing Sample_Name: 288
Start with UASG: 285
Other IDs: 3


In [88]:
other_examples = (col[~mask_uasg]
                  .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
                  .dropna()
                  .unique()[:25])
print("Examples of non-UASG IDs:", other_examples)


Examples of non-UASG IDs: ['NA10837' 'NA12239' 'NA12146']


In [85]:
# --- extract clean UASG IDs from the main df (which has 'UASG' column) ---
cohort_ids = (df_csv["UASG"]
              .astype(str)
              .str.strip()
              .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
              .dropna()
              .unique())

print("Unique UASG in cohort df:", len(cohort_ids))

Unique UASG in cohort df: 229


In [90]:
# --- compare sets ---
snp_only = sorted(set(snp_ids) - set(cohort_ids))
cohort_only = sorted(set(cohort_ids) - set(snp_ids))
both = len(set(snp_ids) & set(cohort_ids))

print("In both:", both)
print("Only in df_snp:", len(snp_only))
print("Only in cohort df:", len(cohort_only))

In both: 226
Only in df_snp: 62
Only in cohort df: 3


In [91]:
# Peek at a few missing IDs
print("Example IDs only in df_snp:", snp_only[:10])
print("Example IDs only in cohort df:", cohort_only[:10])

Example IDs only in df_snp: ['NA10837', 'NA12146', 'NA12239', 'UASG-0002', 'UASG-0022', 'UASG-0030', 'UASG-0040', 'UASG-0043', 'UASG-0050', 'UASG-0051']
Example IDs only in cohort df: ['UASG-0129', 'UASG-0191', 'UASG-1168']


In [92]:
# Optional: nice summary table
summary = pd.DataFrame({
    "Status": ["In both", "Only in df_snp", "Only in cohort df"],
    "Count":  [both, len(snp_only), len(cohort_only)]
})
display(summary)

,Status,Count
0,In both,226
1,Only in df_snp,62
2,Only in cohort df,3
